In [3]:
# get table
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
utils.fetch_data_from_postgres_via_psycopg2(
  """
  SELECT table_name
  FROM information_schema.tables
  WHERE table_schema = 'public';
"""
)
# user_id: [1, 6040]
# movie_id: [1, 3706]

,table_name
0,tpcxai_lineitem_serving
1,tpcxai_financial_account_serving
2,tpcxai_product_rating_training
3,tpcxai_order_returns_training
4,tpcxai_order_training
5,tpcxai_product_training
6,tpcxai_financial_transactions_training
7,tpcxai_product_rating_serving
8,tpcxai_order_returns_serving
9,tpcxai_order_serving


In [1]:
# template 4
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
import h5py
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder
from surprise import SVD
from surprise import Dataset
from surprise.reader import Reader

df_train = utils.fetch_data_from_postgres_via_psycopg2("""
  select * from tpcxai_product_rating_training;
""")
reader = Reader()
data_train = Dataset.load_from_df(
    df_train[['user_id', 'product_id', 'rating']], reader)
svd = SVD()
trainset = data_train.build_full_trainset()
model = svd.fit(trainset)
# SVD has the following parameters:
print("bu: ", model.bu.shape)
print("bi: ", model.bi.shape)
print("pu: ", model.pu.shape)
print("qi: ", model.qi.shape)

with h5py.File("/home/velox/resources/model/tpcxai_sf1/final/velox/tpcxai_template4_svd.h5", 'w') as f:
  f.create_dataset('bu', data=model.bu)
  f.create_dataset('bi', data=model.bi)
  f.create_dataset('pu', data=model.pu)
  f.create_dataset('qi', data=model.qi)

2025-06-26 04:40:43.020163: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-26 04:40:43.062600: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-26 04:40:43.062634: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-26 04:40:43.063946: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-26 04:40:43.070875: I tensorflow/core/platform/cpu_feature_guar

bu:  (7071,)
bi:  (6818,)
pu:  (7071, 100)
qi:  (6818, 100)


In [2]:
# testing part
df_test = utils.fetch_data_from_postgres_via_psycopg2("""
  select * from tpcxai_product_rating_serving
  join tpcxai_product_serving on product_id = p_product_id
  join tpcxai_customer_serving on user_id = c_customer_sk;
""")

preds = []
for _, row in tqdm(df_test.iterrows(), total=df_test.shape[0]):
    user_id = row['user_id']
    movie_id = row['product_id']
    pred_rating = model.predict(user_id, movie_id).est
    preds.append(pred_rating)

Error: relation "tpcxai_customer_serving" does not exist
LINE 4:   join tpcxai_customer_serving on user_id = c_customer_sk;
               ^



UnboundLocalError: local variable 'df' referenced before assignment

In [1]:
# template 5
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
import h5py
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder
from surprise import SVD
from surprise import Dataset
from surprise.reader import Reader
from collections import Counter
from transformers import BertTokenizer, RobertaTokenizer

df_train = utils.fetch_data_from_postgres_via_psycopg2("""
  select * from tpcxai_review_training;
""")

# Load tokenizer from vocab.json
# tokenizer = RobertaTokenizer(vocab_file="/home/velox/resources/model/tokenizer/roberta.json")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
tokenizer.save_pretrained("./my_tokenizer")
print("Tokenizer vocab size:", tokenizer.vocab_size)


2025-06-23 01:10:39.271028: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-23 01:10:39.314426: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-23 01:10:39.314467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-23 01:10:39.315702: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 01:10:39.322611: I tensorflow/core/platform/cpu_feature_guar

Tokenizer vocab size: 50265


In [2]:
X_features = np.zeros((df_train.shape[0], tokenizer.vocab_size))
vocab_size = tokenizer.vocab_size
for idx, row in tqdm(df_train.iterrows(), total=df_train.shape[0]):
    text = row['text']
    tokens = tokenizer(text)
    token_ids = tokens['input_ids']
    token_counter = Counter(token_ids)
    token_freq_vector = np.zeros(vocab_size, dtype=np.float32)    
    for token_id, freq in token_counter.items():
      if token_id < vocab_size:
        token_freq_vector[token_id] = freq
    token_freq_vector = token_freq_vector / np.sum(token_freq_vector)
    
  y_labels = df_train['spam'].values

  0%|          | 0/134349 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (516 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(vocab_size,)),
    tf.keras.layers.Dense(2048, activation='relu'),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss=BinaryCrossentropy(), metrics=['accuracy'])
# model.fit(X_features, y_labels, epochs=5, batch_size=1024, validation_split=0.2)

2025-06-23 01:13:30.876872: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20718 MB memory:  -> device: 0, name: NVIDIA A10, pci bus id: 0000:65:00.0, compute capability: 8.6
2025-06-23 01:13:30.878630: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 20718 MB memory:  -> device: 1, name: NVIDIA A10, pci bus id: 0000:ca:00.0, compute capability: 8.6


In [ ]:
# testing part
df_test = utils.fetch_data_from_postgres_via_psycopg2("""
  select * from tpcxai_review_serving
  limit 1000;
""")

X_test_features = np.zeros((df_test.shape[0], tokenizer.vocab_size))

for idx, row in tqdm(df_test.iterrows(), total=df_test.shape[0]):
    text = row['text']
    tokens = tokenizer(text)
    token_ids = tokens['input_ids']
    token_counter = Counter(token_ids)
    token_freq_vector = np.zeros(vocab_size, dtype=np.float32)    
    for token_id, freq in token_counter.items():
      if token_id < vocab_size:
        token_freq_vector[token_id] = freq
    token_freq_vector = token_freq_vector / np.sum(token_freq_vector)
    X_test_features[idx] = token_freq_vector

model.predict(X_test_features, batch_size=256)

  0%|          | 0/1000 [00:00<?, ?it/s]

In [1]:
# template 6
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder


# Use case 8, trainig query
df_training_data = utils.fetch_data_from_postgres_via_psycopg2("""
SELECT 
        o_order_id,
        department,
        quantity,
        SUM(quantity) AS scan_count,                -- Equivalent to np.sum(x)
        MIN(EXTRACT(DOW FROM date)) AS weekday,     -- Equivalent to np.min(x) for weekday
        MIN(trip_type) AS trip_type                 -- Equivalent to np.min(x) for trip_type
    FROM tpcxai_order_training 
    JOIN tpcxai_lineitem_training ON o_order_id = li_order_id 
    JOIN tpcxai_product_training ON li_product_id = p_product_id
    GROUP BY o_order_id, date, department, quantity
""")

le_department = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_training_data['department_encoded'] = le_department.fit_transform(df_training_data[['department']])
le_trip_type = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_training_data['trip_type_encoded'] = le_trip_type.fit_transform(df_training_data[['trip_type']])

X_features = df_training_data[['quantity', 'scan_count', 'weekday', 'department_encoded']].values.astype(float)
y = df_training_data['trip_type_encoded'].values.astype(float)
num_y = len(np.unique(y))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_features.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_y, activation='softmax')  # Output layer for trip_type
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
X_train, X_val, y_train, y_val = train_test_split(X_features, y, test_size=0.2, random_state=42)
# model.fit(X_train, y_train, epochs=10, batch_size=2048, validation_data=(X_val, y_val))

2025-06-25 23:31:59.174812: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-25 23:31:59.217713: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-25 23:31:59.217753: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-25 23:31:59.219091: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-25 23:31:59.226440: I tensorflow/core/platform/cpu_feature_guar

In [5]:
print(X_features.shape, num_y)

(16346475, 4) 38


In [2]:
# testing part
df_test_data = utils.fetch_data_from_postgres_via_psycopg2("""
SELECT 
        o_order_id,
        department,
        quantity,
        SUM(quantity) AS scan_count,                -- Equivalent to np.sum(x)
        MIN(EXTRACT(DOW FROM date)) AS weekday     -- Equivalent to np.min(x) for weekday
    FROM tpcxai_order_serving 
    JOIN tpcxai_lineitem_serving ON o_order_id = li_order_id 
    JOIN tpcxai_product_serving ON li_product_id = p_product_id
    GROUP BY o_order_id, date, department, quantity
""")

df_test_data['department_encoded'] = le_department.transform(df_test_data[['department']])
X_infer = df_test_data[['quantity', 'scan_count', 'weekday', 'department_encoded']].values.astype(float)
y_pred = model.predict(X_infer, batch_size=2048)


806/806 [==============================] - 2s 1ms/step


In [3]:
# encoder 
print("Ordinal Encoder for Department:", le_department.categories_)
print("Ordinal Encoder for Trip Type:", le_trip_type.categories_)

Ordinal Encoder for Department: [array(['AUTOMOTIVE', 'BATH AND SHOWER', 'BEAUTY', 'BEDDING', 'BOYS WEAR',
       'CANDY, TOBACCO, COOKIES', 'CELEBRATION', 'COMM BREAD',
       'COOK AND DINE', 'DAIRY', 'DSD GROCERY', 'ELECTRONICS',
       'FABRICS AND CRAFTS', 'FINANCIAL SERVICES', 'FROZEN FOODS',
       'GIRLS WEAR, 4-6X  AND 7-14', 'GROCERY DRY GOODS', 'HARDWARE',
       'HOME DECOR', 'HOME MANAGEMENT', 'HORTICULTURE AND ACCESS',
       'HOUSEHOLD CHEMICALS/SUPP', 'HOUSEHOLD PAPER GOODS',
       'IMPULSE MERCHANDISE', 'INFANT APPAREL',
       'INFANT CONSUMABLE HARDLINES', 'JEWELRY AND SUNGLASSES',
       'LADIESWEAR', 'LAWN AND GARDEN', 'LIQUOR,WINE,BEER',
       'MEAT - FRESH & FROZEN', 'MEDIA AND GAMING', 'MENS WEAR',
       'OFFICE SUPPLIES', 'PAINT AND ACCESSORIES', 'PERSONAL CARE',
       'PETS AND SUPPLIES', 'PHARMACY OTC', 'PHARMACY RX',
       'PLAYERS AND ELECTRONICS', 'PRODUCE', 'SERVICE DELI', 'SHOES',
       'SPORTING GOODS', 'TOYS', 'WIRELESS'], dtype=object)]
Ordinal 

In [12]:
# template 7
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder

# load data
# df_user = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_user""")
# df_movie = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_movie""")
# df_rating = utils.fetch_data_from_postgres_via_psycopg2("""SELECT * from movielens_rating""")

# trainig query
df_training_data = utils.fetch_data_from_postgres_via_psycopg2("""
select transaction_id, EXTRACT(HOUR FROM time) / 23 as business_hour_norm, amount / transaction_limit as amount_norm, is_fraud
from tpcxai_financial_account_training join tpcxai_financial_transactions_training on fa_customer_sk=sender_id
""")

In [13]:
X_features = df_training_data[['business_hour_norm', 'amount_norm']].values.astype(float)
y = df_training_data['is_fraud'].values.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)
# train DNN for fraud detection
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# model.fit(X_train, y_train, epochs=10, batch_size=512, validation_data=(X_test, y_test))

In [15]:
# testing part
df_test_data = utils.fetch_data_from_postgres_via_psycopg2(
    """
select transaction_id, EXTRACT(HOUR FROM time) / 23 as business_hour_norm, amount / transaction_limit as amount_norm, time, amount, sender_id, c_birth_day, c_birth_month, c_birth_year, c_birth_country
from tpcxai_financial_account_serving 
join tpcxai_financial_transactions_serving on fa_customer_sk=sender_id
join tpcxai_customer_serving ON fa_customer_sk = c_customer_sk
"""
)
X_infer = df_test_data[["business_hour_norm", "amount_norm"]].values.astype(float)
y_pred = model.predict(X_infer, batch_size=512)

1585/1585 [==============================] - 1s 886us/step


In [1]:
# template 8
import pandas as pd
import numpy as np
import sys
sys.path.append('/home/velox/db-ml/baseline')
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder

df_training_data = utils.fetch_data_from_postgres_via_psycopg2("""
select store, department, li_order_id, price, quantity,
EXTRACT(WEEK FROM date) AS week,
EXTRACT(MONTH FROM date) AS month,
CASE 
    WHEN EXTRACT(WEEK FROM date) > 50 AND EXTRACT(MONTH FROM date) = 1 
    THEN EXTRACT(YEAR FROM date) - 1
    ELSE EXTRACT(YEAR FROM date)
END AS year,
quantity * price as row_price
from tpcxai_order_training join tpcxai_lineitem_training on o_order_id=li_order_id
join tpcxai_product_training on li_product_id=p_product_id
""")

2025-06-19 05:52:41.567345: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 05:52:41.609768: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-19 05:52:41.609805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-19 05:52:41.611016: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-19 05:52:41.618286: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
grouped = df_training_data.groupby(['store', 'department', 'year', 'week'])['row_price'].sum().reset_index()
grouped = grouped.rename(index=str, columns={'store': 'Store', 'department': 'Dept', 'date': 'Date', 'row_price': 'Weekly_Sales'})
grouped['num_of_week'] = (grouped['year'].astype(int) - 2010) * 52 + grouped['week'].astype(int) - 1
le_store = LabelEncoder()
le_dept = LabelEncoder()

grouped['Store'] = le_store.fit_transform(grouped['Store'])
grouped['Dept'] = le_dept.fit_transform(grouped['Dept'])
min_num_of_week = 0
max_num_of_week = 52*3
grouped['num_of_week'] = (grouped['num_of_week'] - 0) / max_num_of_week
X_features = grouped[['Store', 'Dept', 'num_of_week']].values
y = grouped['Weekly_Sales'].values
y_min = y.min()
y_max = y.max()
y = (y - y_min) / (y_max - y_min)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(3,)),
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=0)
model.fit(X_train, y_train, epochs=10, batch_size=256, validation_data=(X_test, y_test))

In [ ]:
# testing part
df_testing_data = utils.fetch_data_from_postgres_via_psycopg2("""
select store, department, num_of_week from tpcxai_store_dept_serving
""")
df_testing_data['store'] = le_store.transform(df_testing_data['store'].values)
df_testing_data['department'] = le_dept.transform(df_testing_data['department'].values)
df_testing_data['num_of_week'] = (df_testing_data['num_of_week'] - 0) / max_num_of_week
X_serve = df_testing_data[['store', 'department', 'num_of_week']].values.astype(float)
y_pred = model.predict(X_serve)
y_pred = y_pred * (y_max - y_min) + y_min

In [7]:
print("Label Encoder for Store:", le_store.classes_)
print("Label Encoder for Department:", le_dept.classes_)

Label Encoder for Store: [ 1  2  3  4  5  6  7  8  9 10 11]
Label Encoder for Department: ['AUTOMOTIVE' 'BATH AND SHOWER' 'BEAUTY' 'BEDDING' 'BOYS WEAR'
 'CANDY, TOBACCO, COOKIES' 'CELEBRATION' 'COMM BREAD' 'COOK AND DINE'
 'DAIRY' 'DSD GROCERY' 'ELECTRONICS' 'FABRICS AND CRAFTS'
 'FINANCIAL SERVICES' 'FROZEN FOODS' 'GIRLS WEAR, 4-6X  AND 7-14'
 'GROCERY DRY GOODS' 'HARDWARE' 'HOME DECOR' 'HOME MANAGEMENT'
 'HORTICULTURE AND ACCESS' 'HOUSEHOLD CHEMICALS/SUPP'
 'HOUSEHOLD PAPER GOODS' 'IMPULSE MERCHANDISE' 'INFANT APPAREL'
 'INFANT CONSUMABLE HARDLINES' 'JEWELRY AND SUNGLASSES' 'LADIESWEAR'
 'LAWN AND GARDEN' 'LIQUOR,WINE,BEER' 'MEAT - FRESH & FROZEN'
 'MEDIA AND GAMING' 'MENS WEAR' 'OFFICE SUPPLIES' 'PAINT AND ACCESSORIES'
 'PERSONAL CARE' 'PETS AND SUPPLIES' 'PHARMACY OTC' 'PHARMACY RX'
 'PLAYERS AND ELECTRONICS' 'PRODUCE' 'SERVICE DELI' 'SHOES'
 'SPORTING GOODS' 'TOYS' 'WIRELESS']


In [ ]:
# template 9
import pandas as pd
import numpy as np
import sys

sys.path.append("/home/velox/db-ml/baseline")
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder
from sklearn.cluster import KMeans

df_training_data = utils.fetch_data_from_postgres_via_psycopg2(
    """

with groups as (
select o_customer_sk, o_order_id, min(EXTRACT(YEAR FROM date)) as invoice_year, sum(or_return_quantity * price) / sum(quantity * price) as ratio                                                                   
from tpcxai_order_training join tpcxai_lineitem_training on o_order_id=li_order_id
join tpcxai_product_training on li_product_id=p_product_id
join tpcxai_order_returns_training on o_order_id=or_order_id
group by o_customer_sk, o_order_id
),

ratio as (
select o_customer_sk, avg(ratio) as avg_return_ratio
from groups
group by o_customer_sk
),

frequency as (select o_customer_sk, avg(num_return)
from (
  select o_customer_sk, invoice_year, count(*) as num_return
  from groups
  group by o_customer_sk, invoice_year
) as yearly_return
group by o_customer_sk
)

select * from
ratio join frequency using (o_customer_sk)
"""
)
min_max_scaler = MinMaxScaler()
X_features = df_training_data[['avg_return_ratio', 'avg']].values.astype(float)
X_features = min_max_scaler.fit_transform(X_features)
kmeans = KMeans(n_clusters=30, random_state=0).fit(X_features)
y_labels = kmeans.labels_

model = tf.keras.Sequential(
    [
        tf.keras.layers.Dense(256, activation="relu", input_shape=(2,)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(30, activation="softmax"),  # Output layer for 30 clusters
    ]
)
model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42
)
# model.fit(X_train, y_train, epochs=10, batch_size=2048, validation_data=(X_val, y_val))

In [ ]:
# testing part
df_test_data = utils.fetch_data_from_postgres_via_psycopg2(
    """

    with groups as (
select o_customer_sk, o_order_id, min(EXTRACT(YEAR FROM date)) as invoice_year, sum(or_return_quantity * price) / sum(quantity * price) as ratio                                                                   
from tpcxai_order_serving join tpcxai_lineitem_serving on o_order_id=li_order_id
join tpcxai_product_serving on li_product_id=p_product_id
join tpcxai_order_returns_serving on o_order_id=or_order_id
group by o_customer_sk, o_order_id
),

ratio as (
select o_customer_sk, avg(ratio) as avg_return_ratio
from groups
group by o_customer_sk
),

frequency as (select o_customer_sk, avg(num_return)
from (
  select o_customer_sk, invoice_year, count(*) as num_return
  from groups
  group by o_customer_sk, invoice_year
) as yearly_return
group by o_customer_sk
)

select * from
ratio join frequency using (o_customer_sk)
"""
)

X_test_features = df_test_data[['avg_return_ratio', 'avg']].values.astype(float)
X_test_features = min_max_scaler.transform(X_test_features)
y_pred = model.predict(X_test_features, batch_size=2048)

In [48]:
print("MinMaxScaler data min:", min_max_scaler.data_min_, "data max:", min_max_scaler.data_max_)
print("KMeans cluster centers:\n", kmeans.cluster_centers_)

MinMaxScaler data min: [0.2 1. ] data max: [ 2.17857143 12.        ]
KMeans cluster centers:
 [[0.21419288 0.03119531]
 [0.30669107 0.22728179]
 [0.25110542 0.36363636]
 [0.23727938 0.09088499]
 [0.25204325 0.22726835]
 [0.33491766 0.03275516]
 [0.2867876  0.09090909]
 [0.17763221 0.09963197]
 [0.25668713 0.18181818]
 [0.27758786 0.31818182]
 [0.40369838 0.11037086]
 [0.30299368 0.4659766 ]
 [0.3301539  0.13636364]
 [0.27745749 0.13636698]
 [0.22704171 0.13637324]
 [0.36871279 0.19348158]
 [0.2743455  0.        ]
 [0.14335284 0.01320307]
 [0.30553181 0.18181818]
 [0.28177035 0.27272727]
 [0.28137665 0.58158996]
 [0.2237054  0.2815085 ]
 [0.25208153 0.42783505]
 [0.31962837 0.37226502]
 [0.52800041 0.04324495]
 [0.27434346 0.04551355]
 [0.33889412 0.09088677]
 [0.20163882 0.19021638]
 [0.40303089 0.01893626]
 [0.34312296 0.28549344]]


In [1]:
# template 10
import pandas as pd
import numpy as np
import sys

sys.path.append("/home/velox/db-ml/baseline")
import utils
import tensorflow as tf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.losses import BinaryCrossentropy
from sklearn.preprocessing import OrdinalEncoder
from sklearn.cluster import KMeans

df_train = utils.fetch_data_from_postgres_via_psycopg2("""
SELECT * 
from tpcxai_product_rating_training                                                       
join tpcxai_customer_training on user_id = c_customer_sk
join tpcxai_product_training on product_id = p_product_id;
""")                                                                                                              

2025-06-23 03:35:15.227269: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-23 03:35:15.270353: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-23 03:35:15.270383: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-23 03:35:15.271841: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 03:35:15.279542: I tensorflow/core/platform/cpu_feature_guar

In [3]:
df_train.head()

,user_id,product_id,rating,c_customer_sk,c_customer_id,c_current_addr_sk,c_first_name,c_last_name,c_preferred_cust_flag,c_birth_day,c_birth_month,c_birth_year,c_birth_country,c_login,c_email_address,c_cluster_id,p_product_id,name,department
0,1,241,1,1,AAAAAAAAAAAAAAAB,19275,Mireielle,Srivastava,N,19,1,1959,FRENCH GUIANA,hyGbpY,Mireielle.Srivastava@vegemail.com,1,241,Scck3I,FABRICS AND CRAFTS
1,2,166,5,2,AAAAAAAAAAAAAAAC,28641,Gigi,Townsel,N,10,8,2000,YEMEN,rC8G,Gigi.Townsel@usa.com,1,166,sc,DSD GROCERY
2,2,167,4,2,AAAAAAAAAAAAAAAC,28641,Gigi,Townsel,N,10,8,2000,YEMEN,rC8G,Gigi.Townsel@usa.com,1,167,wiPRwaRDXj,CELEBRATION
3,2,242,5,2,AAAAAAAAAAAAAAAC,28641,Gigi,Townsel,N,10,8,2000,YEMEN,rC8G,Gigi.Townsel@usa.com,1,242,r,OFFICE SUPPLIES
4,2,110,5,2,AAAAAAAAAAAAAAAC,28641,Gigi,Townsel,N,10,8,2000,YEMEN,rC8G,Gigi.Townsel@usa.com,1,110,WbVondnF1LtuVH,IMPULSE MERCHANDISE


In [11]:
department_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df_train['department_encoded'] = department_encoder.fit_transform(df_train[['department']])
min_max_scaler = MinMaxScaler()
df_train[["c_birth_day_encoded", "c_birth_month_encoded", "c_birth_year_encoded"]] = min_max_scaler.fit_transform(df_train[["c_birth_day", "c_birth_month", "c_birth_year"]])
category_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

rating_scalaer = MinMaxScaler()
df_train['rating_scaled'] = rating_scalaer.fit_transform(df_train[['rating']])

X_features = df_train[['department_encoded', 'c_birth_day_encoded', 'c_birth_month_encoded', 'c_birth_year_encoded']].values.astype(float)
y = df_train['rating_scaled'].values.astype(float)
X_train, X_val, y_train, y_val = train_test_split(X_features, y, test_size=0.2, random_state=42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(256, activation='relu', input_shape=(X_features.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
# model.fit(X_train, y_train, epochs=10, batch_size=2048, validation_data=(X_val, y_val))

2025-06-23 04:17:11.811648: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20718 MB memory:  -> device: 0, name: NVIDIA A10, pci bus id: 0000:65:00.0, compute capability: 8.6
2025-06-23 04:17:11.813382: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 20718 MB memory:  -> device: 1, name: NVIDIA A10, pci bus id: 0000:ca:00.0, compute capability: 8.6


In [13]:
df_test = utils.fetch_data_from_postgres_via_psycopg2("""
SELECT *
from tpcxai_product_rating_serving
join tpcxai_customer_serving on user_id = c_customer_sk
join tpcxai_product_serving on product_id = p_product_id
""")

df_test['department_encoded'] = department_encoder.transform(df_test[['department']])
df_test[["c_birth_day_encoded", "c_birth_month_encoded", "c_birth_year_encoded"]] = min_max_scaler.transform(df_test[["c_birth_day", "c_birth_month", "c_birth_year"]])
X_infer = df_test[['department_encoded', 'c_birth_day_encoded', 'c_birth_month_encoded', 'c_birth_year_encoded']].values.astype(float)
y_pred = model.predict(X_infer, batch_size=2048)

1/1 [==============================] - 0s 291ms/step
